---
# `Multi Query Retriever`
---

### Intro
- Query: How can I stay healthy ?
- break it down into mulitple queries: 
  - What should I eat?
  - How often should I exercise?
  - How can I manage stress ?

- Sometimes a single query, might not capture all the ways of information as phrase in your documents 
- a simple similarity search might not provide with relevant documents
- As in that case , healthy, food, exercise, stress etc
- That how Multi Query Retriever helps:  for the given original query - How can I stay healthy  --> provide to the LLM model
- LLM Model: it break the original query into multiple sub query
  - What are the best food to eat for good health
  - how often should I exercise
  - What lifestyle habit
- All for sub query, they will be multiple retriever --> they fetch all the details like top 5 food, top 5 excerise , top 5 stress relieving habits
- then remove duplicate information from them and suggest the best 10 documents to the Original Query.

# `Detailed Notes`

# Multi-Query Retriever in LangChain

**Multi-Query Retriever** is a retrieval strategy used in RAG where **one user question is transformed into multiple alternative queries**, and each query is used to retrieve relevant documents.

The main goal is:

> **Improve retrieval recall by searching for the same information from multiple perspectives.**

---

# 1. Why Do We Need Multi-Query Retrieval?

Consider the user asks:

> **"How does attention work in Transformers?"**

Your vector database might contain:

```text
Document 1 → Self-attention mechanism
Document 2 → Query, Key, Value
Document 3 → Multi-head attention
Document 4 → Attention scores
Document 5 → Transformer architecture
```

A single query:

```text
"How does attention work in Transformers?"
```

might retrieve:

```text
Document 1
Document 2
Document 4
```

But perhaps it misses Document 3 because its wording is different.

Multi-Query Retriever solves this by generating multiple versions of the question.

---

# 2. Basic Idea

Instead of:

```text
User Query
    ↓
One Search
    ↓
Documents
```

we do:

```text
User Query
    ↓
LLM generates multiple queries
    ↓
┌────────────┬────────────┬────────────┐
│ Query 1    │ Query 2    │ Query 3    │
└─────┬──────┴─────┬──────┴─────┬──────┘
      ↓            ↓            ↓
   Search       Search       Search
      ↓            ↓            ↓
   Docs         Docs         Docs
      └────────────┼────────────┘
                   ↓
             Combine Results
                   ↓
             Final Documents
                   ↓
                  LLM
```

---

# 3. Example

Original question:

> **"How does attention work in Transformers?"**

The Multi-Query Retriever may ask an LLM to generate:

### Query 1

> "How does self-attention work in Transformer models?"

### Query 2

> "What are Query, Key, and Value in Transformer attention?"

### Query 3

> "How are attention scores calculated in Transformers?"

### Query 4

> "What is the purpose of multi-head attention?"

Now retrieval searches all of them.

---

# 4. Why Multiple Queries Help

Different queries can retrieve different documents.

```text
Query 1
   ↓
D1, D2, D5

Query 2
   ↓
D2, D3, D7

Query 3
   ↓
D4, D5, D8

Query 4
   ↓
D3, D6, D9
```

The retriever combines the results:

```text
D1
D2
D3
D4
D5
D6
D7
D8
D9
```

Then duplicate documents can be removed.

The result is a broader set of relevant information.

---

# 5. The Main Problem It Solves

Multi-Query Retrieval primarily addresses **query-document vocabulary mismatch** and retrieval recall.

For example:

User says:

```text
"How does a model remember previous words?"
```

The document says:

```text
"Long-range dependencies in sequential data..."
```

A semantic retriever may understand the relationship, but depending on the embedding and corpus, the original query may not retrieve the best chunk.

Alternative queries could be:

```text
"How are long-range dependencies handled?"
"What mechanisms preserve information across a sequence?"
"How does a neural network retain information from previous tokens?"
```

Each gives the retrieval system another opportunity to find the right documents.

---

# 6. Multi-Query Retriever vs Normal Retriever

### Normal Retriever

```text
Question
   ↓
Embedding/Search
   ↓
Top K Documents
```

### Multi-Query Retriever

```text
Question
   ↓
LLM
   ↓
Multiple Queries
   ↓
Multiple Searches
   ↓
Combine Results
   ↓
Relevant Documents
```

---

# 7. Multi-Query Retriever vs MMR

You just learned MMR, so this distinction is important.

### MMR

MMR asks:

> **"Among relevant documents, which ones are useful and not redundant?"**

```text
One Query
   ↓
Candidate Documents
   ↓
Relevance + Diversity
   ↓
Final Documents
```

### Multi-Query

Multi-Query asks:

> **"What different ways can I express this question so I can retrieve more relevant documents?"**

```text
One Query
   ↓
Multiple Query Variations
   ↓
Multiple Searches
   ↓
Combined Documents
```

So:

|                      | MMR             | Multi-Query                  |
| -------------------- | --------------- | ---------------------------- |
| Main goal            | Diversity       | Recall                       |
| Input queries        | One             | Multiple                     |
| Uses LLM to rewrite? | Not necessarily | Yes                          |
| Reduces redundancy   | Yes             | Not its primary purpose      |
| Expands search space | No              | Yes                          |
| Typical benefit      | Diverse context | Find more relevant documents |

---

# 8. Multi-Query Retriever Architecture

```text
                     USER
                       │
                       ▼
                Original Question
                       │
                       ▼
                 Query-Generating LLM
                       │
            ┌──────────┼──────────┐
            ▼          ▼          ▼
         Query 1    Query 2    Query 3
            │          │          │
            ▼          ▼          ▼
        Retriever  Retriever  Retriever
            │          │          │
            ▼          ▼          ▼
          Docs       Docs       Docs
            │          │          │
            └──────────┼──────────┘
                       ▼
                 Deduplicate
                       │
                       ▼
               Relevant Documents
                       │
                       ▼
                  Generation LLM
                       │
                       ▼
                    Answer
```

There can be additional ranking or filtering after retrieval depending on the architecture.

---

# 9. Multi-Query Retrieval in LangChain

LangChain provides a `MultiQueryRetriever`.

A typical implementation is:

```python
from langchain.retrievers.multi_query import MultiQueryRetriever
```

You need:

1. A base retriever
2. An LLM

For example:

```python
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)
```

Then:

```python
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)
```

Now:

```python
docs = multi_query_retriever.invoke(
    "How does attention work in Transformers?"
)
```

---

# 10. What Happens Internally?

When you execute:

```python
docs = multi_query_retriever.invoke(
    "How does attention work in Transformers?"
)
```

conceptually:

### Step 1 — Original query

```text
How does attention work in Transformers?
```

### Step 2 — LLM generates alternatives

```text
Query 1:
How does self-attention work?

Query 2:
What are Query, Key, and Value?

Query 3:
How are attention scores calculated?
```

### Step 3 — Each query is sent to the base retriever

```text
Query 1 → Vector Store → Documents
Query 2 → Vector Store → Documents
Query 3 → Vector Store → Documents
```

### Step 4 — Results are combined

```text
Documents from Query 1
+
Documents from Query 2
+
Documents from Query 3
```

### Step 5 — Duplicates are removed

You get a broader set of unique documents.

---

# 11. Important: The LLM Is Used for Query Generation

This is the defining feature.

Normal vector retrieval:

```text
Question
 ↓
Embedding Model
 ↓
Vector Search
```

Multi-query retrieval:

```text
Question
 ↓
LLM
 ↓
Multiple Queries
 ↓
Embedding/Search
 ↓
Documents
```

Therefore:

> **Multi-Query Retrieval uses an LLM to expand one query into multiple search queries.**

---

# 12. Why Not Just Use a Better Embedding Model?

A better embedding model can improve retrieval, but query rewriting solves a different problem.

Consider:

```text
User:
"Why does my model forget old information?"
```

The knowledge base might contain:

```text
"Long-term dependencies"
"Vanishing gradients"
"Memory retention"
"Recurrent state"
```

Different query formulations can expose different concepts:

```text
"What causes models to lose information from earlier tokens?"

"What are long-term dependency problems in neural networks?"

"How do neural networks preserve information over long sequences?"
```

This increases the chances of retrieving relevant chunks.

---

# 13. Multi-Query Improves Recall

This is one of the most important terms.

### Recall

In retrieval, recall broadly asks:

> **How much of the relevant information did we manage to retrieve?**

Suppose the knowledge base contains 10 relevant documents.

Normal retrieval finds:

```text
4 / 10
```

Recall:

```text
40%
```

Multi-query might find:

```text
8 / 10
```

Recall:

```text
80%
```

The actual improvement depends heavily on the dataset, query generation quality, base retriever, and ranking strategy.

---

# 14. But There Is a Cost

Multi-query isn't automatically better.

If one query costs:

```text
1 retrieval operation
```

and you generate 5 queries:

```text
5 retrieval operations
```

You increase:

* Latency
* Compute
* Embedding/search operations
* LLM query-generation cost

Architecture:

```text
Normal:

Question
  ↓
1 Search
  ↓
Answer


Multi-Query:

Question
  ↓
LLM
  ↓
5 Queries
  ↓
5 Searches
  ↓
Combine
  ↓
Answer
```

So you trade:

```text
Higher recall
      ↕
Higher cost + latency
```

---

# 15. When Should You Use Multi-Query?

It is particularly useful when the user's query can be interpreted in multiple ways.

### Example 1 — Complex question

> "How does a Transformer understand the relationship between words?"

Possible interpretations:

```text
Self-attention
Context
Q/K/V
Attention scores
Positional encoding
```

Multi-query can search multiple formulations.

---

### Example 2 — Ambiguous terminology

```text
"What is memory in neural networks?"
```

Could refer to:

```text
RNN hidden state
LSTM memory cell
External memory
Context window
Model parameters
```

Multiple queries can improve coverage.

---

### Example 3 — Poorly worded user query

```text
"model forget previous thing why?"
```

The LLM can transform it into better search queries.

---

# 16. When Should You NOT Use It?

If the query is already highly precise:

> "What is the default learning rate in this configuration?"

Generating several variations may provide little benefit.

Normal retrieval may be sufficient:

```text
Question
 ↓
Vector Search
 ↓
Relevant Chunk
```

Also avoid unnecessary multi-query expansion when latency and cost are strict constraints.

---

# 17. Multi-Query in a PDF Chatbot

Suppose you're building your PDF chatbot.

Normal:

```text
User:
"Explain attention."

       ↓

Embedding

       ↓

Chroma

       ↓

Top 5 chunks

       ↓

LLM
```

Multi-query:

```text
User:
"Explain attention."

       ↓

Query LLM

       ↓
 ┌──────────────┬─────────────────┬────────────────┐
 │              │                 │
 ▼              ▼                 ▼
"attention"   "self-attention"   "attention in
                                  Transformers"
 │              │                 │
 ▼              ▼                 ▼
Chroma         Chroma            Chroma
 │              │                 │
 └──────────────┼─────────────────┘
                ▼
         Combine Results
                ↓
          Unique Documents
                ↓
               LLM
```

---

# 18. Multi-Query + MMR

These techniques can also be combined.

For example:

```text
User Query
    ↓
Multi-Query
    ↓
Query 1 ──┐
Query 2 ──┼──→ Candidate Documents
Query 3 ──┘
                 ↓
                MMR
                 ↓
       Relevant + Diverse Documents
                 ↓
                LLM
```

Here:

* **Multi-Query** improves **recall**
* **MMR** improves **diversity / reduces redundancy**

This can be powerful, but it also increases complexity and latency.

---

# 19. Multi-Query vs Query Rewriting

They're related but not identical.

### Query Rewriting

Usually:

```text
Original Query
      ↓
Better Query
      ↓
Retriever
```

One improved query.

### Multi-Query

```text
Original Query
      ↓
Query 1
Query 2
Query 3
      ↓
Multiple Retrievals
```

Multiple alternative queries.

---

# 20. Interview Answer

If asked:

> **What is Multi-Query Retriever in LangChain?**

A strong answer:

> **Multi-Query Retriever is a LangChain retrieval strategy that uses an LLM to generate multiple alternative formulations of a user's query. It sends those queries to the underlying retriever, combines the resulting documents, and typically removes duplicates. Its primary purpose is to improve retrieval recall by searching for the required information from multiple perspectives.**

---

# 21. Final Mental Model

Remember the difference between the retrieval techniques you've learned:

```text
Similarity Search
        ↓
"Find documents most similar to my query."


MMR
        ↓
"Find relevant documents
 while reducing redundancy."


Multi-Query
        ↓
"Rewrite my question in multiple ways
 and search each version."
```

And in RAG:

```text
                    USER QUERY
                         │
                         ▼
                  Multi-Query LLM
                         │
            ┌────────────┼────────────┐
            ▼            ▼            ▼
         Query 1      Query 2      Query 3
            │            │            │
            └────────────┼────────────┘
                         ▼
                  Base Retriever
                         │
                         ▼
                  Relevant Documents
                         │
                         ▼
                      Context
                         │
                         ▼
                        LLM
                         │
                         ▼
                      Answer
```

**Core takeaway:**

> **Multi-Query Retriever expands one user question into multiple search queries to increase the probability of finding all relevant information.**
